In [8]:
%pip install -q timm pillow pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
from pathlib import Path

ROOT = Path('/home/boibhai/GitHub/nematode-detection')
TRAIN_DIR = ROOT / 'dataset_inema' / 'train'
VAL_DIR = ROOT / 'dataset_inema' / 'val'
PRETRAINED_DIR = ROOT / 'pretrained_models'
TRAINED_DIR = ROOT / 'trained_models'
PRETRAINED_DIR.mkdir(exist_ok=True)
TRAINED_DIR.mkdir(exist_ok=True)

# torchvision and timm store downloaded pretrained weights here, not in a temporary kernel cache.
os.environ['TORCH_HOME'] = str(PRETRAINED_DIR)
os.environ['HF_HOME'] = str(PRETRAINED_DIR / 'huggingface')

import gc
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGE_SIZE = 224
BATCH_SIZE = 2       # VGG16 fits safely on this 3.68 GB GPU at this batch size.
EPOCHS = 10
LR = 1e-4
print('Device:', DEVICE)

Device: cuda


In [10]:
# Create five rotated copies of every original training image.
# Existing *_rot_ copies are ignored, so rerunning this cell does not make duplicates.
random.seed(42)
extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
originals = [p for p in TRAIN_DIR.rglob('*') if p.suffix.lower() in extensions and '_rot_' not in p.stem]

for source in originals:
    with Image.open(source) as image:
        image = ImageOps.exif_transpose(image).convert('RGB')
        for copy_number in range(1, 2):
            target = source.with_name(f'{source.stem}_rot_{copy_number}.jpg')
            if not target.exists():
                angle = random.uniform(-180, 180)
                image.rotate(angle, resample=Image.Resampling.BICUBIC).save(target, quality=95)

print(f'Original images: {len(originals)}')
print(f'Training images after rotation: {sum(1 for p in TRAIN_DIR.rglob("*") if p.suffix.lower() in extensions)}')
print(f'Validation images unchanged: {sum(1 for p in VAL_DIR.rglob("*") if p.suffix.lower() in extensions)}')

Original images: 2217
Training images after rotation: 4434
Validation images unchanged: 552


In [12]:
# Run once. Downloads all three original ImageNet weights into pretrained_models when it is empty.
if not any(PRETRAINED_DIR.iterdir()):
    models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
    timm.create_model('vgg16.tv_in1k', pretrained=True)
    timm.create_model('resnet50', pretrained=True)
    print('Downloaded pretrained models to:', PRETRAINED_DIR)
else:
    print('Using pretrained models already in:', PRETRAINED_DIR)

Using pretrained models already in: /home/boibhai/GitHub/nematode-detection/pretrained_models
